# 作业02：列表，字典和类

| 项目 | 内容 |
|---|---|
| 姓名 | 彭伟明 |
| 学号 | 20244080323 |
| 运行环境 | conda 独立环境 `python-a`（Python 3.12.15） |
| 作业日期 | 2026-10-20 |

本 Notebook 依次完成「字符串的乐趣」「列表的乐趣」「作为向量的元组」「更多向量的乐趣」四部分。每道题先用 Markdown 说明思路，再用代码单元格实现并测试。

## 1.1 第一个代码框：身份与环境记录

该代码框输出当前 conda 环境名、机器指纹与唯一运行 ID。

In [1]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244080323

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]        {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

[Conda 环境] python-a
[学号]        20244080323
[机器指纹] 6bba46cdb724809d
[RUN_ID]   20244080323-6bba46cdb724809d


---

## 2. 字符串的乐趣

### 2-1 `is_palindrome(s)`

思路：先把字符串规范化——**去掉所有非字母数字字符**（空格、标点都被过滤掉，正好覆盖题目给的 `Was it a car or a cat I saw?` 这类例子），再统一转小写，最后与它的逆序比较。

`[::-1]` 是切片写法，返回反序的新字符串；规范化用 `str.isalnum()` 配合 `join`。

In [2]:
def is_palindrome(s):
    cleaned = "".join(ch.lower() for ch in s if ch.isalnum())
    return cleaned == cleaned[::-1]

tests = [
    "level",
    "kayak",
    "pop",
    "tacocat",
    "T A C O cat",
    "rats live on no evil star",
    "Was it a car or a cat I saw?",
    "hello",
    "Python",
]

for s in tests:
    print(f"{s!r:<40} → {is_palindrome(s)}")

'level'                                  → True
'kayak'                                  → True
'pop'                                    → True
'tacocat'                                → True
'T A C O cat'                            → True
'rats live on no evil star'              → True
'Was it a car or a cat I saw?'           → True
'hello'                                  → False
'Python'                                 → False


说明：`tacocat` 与 `T A C O cat` 都返回 `True`，说明大小写和空格确实被忽略了；`Was it a car or a cat I saw?` 中的问号也被 `isalnum()` 过滤掉了。

### 2-2 `is_abecedarian(s)`

思路：去掉空格、转小写，然后检查**相邻两个字符是否始终非递减**。允许重复字母，所以判断条件是 `<=` 而不是 `<`。

用 `all()` 加生成器表达式，遍历一次即可；空串和单字符会让生成器为空，`all()` 返回 `True`，符合直觉。

In [3]:
def is_abecedarian(s):
    letters = s.replace(" ", "").lower()
    return all(letters[i] <= letters[i + 1] for i in range(len(letters) - 1))

tests = ["adder", "beet", "dog", "cat", "abcd efgh xyz", "a", "aabbcc", "ba"]

for s in tests:
    print(f"{s!r:<16} → {is_abecedarian(s)}")

'adder'          → True
'beet'           → True
'dog'            → False
'cat'            → False
'abcd efgh xyz'  → True
'a'              → True
'aabbcc'         → True
'ba'             → False


### 2-3 `remove_vowels(s)`

思路：遍历字符串，只保留不属于 `aeiou` 的字符。题目只列了小写元音，但输入若是 `'Audio'` 这样的大写开头，按字面只删小写会得到 `'Adjio'` 之类的结果，不符合「去掉所有元音」的本意，所以这里统一先转小写再判断。

In [4]:
def remove_vowels(s):
    vowels = "aeiou"
    return "".join(ch for ch in s if ch.lower() not in vowels)

tests = ["cat", "audio", "aeiou", "xyz", "Hello Python", "AEIOU"]

for s in tests:
    print(f"{s!r:<16} → {remove_vowels(s)!r}")

'cat'            → 'ct'
'audio'          → 'd'
'aeiou'          → ''
'xyz'            → 'xyz'
'Hello Python'   → 'Hll Pythn'
'AEIOU'          → ''


`remove_vowels('aeiou')` 返回空字符串 `''`，说明全部由元音组成的输入也能正确处理。

---

## 3. 列表的乐趣

### 3-1 `list_reverse`

要求输入不是列表时引发「适当的错误」。`AttributeError` 也能间接报错，但语义不准确；这里显式检查类型并抛出 `TypeError`，附上实际类型，便于调用方定位问题。

反转用切片 `lst[::-1]`，返回**新列表**，原列表保持不变。

In [5]:
def list_reverse(lst):
    if not isinstance(lst, list):
        raise TypeError(f"list_reverse() 需要列表，得到 {type(lst).__name__}")
    return lst[::-1]

original = [1, 2, 3]
reversed_list = list_reverse(original)
print("输入      :", original)
print("返回      :", reversed_list)
print("原列表未被修改？", original == [1, 2, 3])

try:
    list_reverse((1, 2, 3))
except TypeError as exc:
    print()
    print("传入元组 →", type(exc).__name__, ":", exc)

输入      : [1, 2, 3]
返回      : [3, 2, 1]
原列表未被修改？ True

传入元组 → TypeError : list_reverse() 需要列表，得到 tuple


### 3-2 `is_sorted(seq)`

要求只能遍历序列一次，所以不能写成 `seq == sorted(seq)`——`sorted()` 既要复制整个序列、又要排序，代价远高于一次扫描。

做法：用下标从前往后走一遍，只要出现 `seq[i] > seq[i + 1]` 就立刻返回 `False`，否则扫完返回 `True`。这段代码只用到 `len()` 和下标访问，因此列表、元组、字符串都能直接使用，体现了 Python 的动态类型：**不需要预先知道 `seq` 的具体类型**。

In [6]:
def is_sorted(seq):
    for i in range(len(seq) - 1):
        if seq[i] > seq[i + 1]:
            return False
    return True

cases = [
    [],
    [1],
    [1, 2, 3, 4],
    [1, 1, 2],
    [3, 2, 1],
    [1, 3, 2],
    (1, 2, 3),
    "abc",
    "acb",
]

for seq in cases:
    print(f"{seq!r:<18} → {is_sorted(seq)}")

[]                 → True
[1]                → True
[1, 2, 3, 4]       → True
[1, 1, 2]          → True
[3, 2, 1]          → False
[1, 3, 2]          → False
(1, 2, 3)          → True
'abc'              → True
'acb'              → False


元组和字符串同样返回正确结果，说明函数只依赖「可索引、元素可比较」这两个特性。复杂度是 $O(n)$、只扫描一次，额外空间为 $O(1)$。

---

## 4. 作为向量的元组

### 4-1 `vec_scalar_mult(t, s)`

要求检查输入类型：`t` 必须是由整数或浮点数组成的元组，`s` 必须是整数或浮点数。不满足时抛出带说明的 `TypeError`。

返回**新元组**（元组不可变，天然不会改到原对象）。

In [7]:
def vec_scalar_mult(t, s):
    if not isinstance(t, tuple):
        raise TypeError(f"第一个参数必须是元组，得到 {type(t).__name__}")
    if isinstance(s, bool) or not isinstance(s, (int, float)):
        raise TypeError(f"标量必须是整数或浮点数，得到 {type(s).__name__}")
    if not all(isinstance(x, (int, float)) and not isinstance(x, bool) for x in t):
        raise TypeError("元组的元素必须都是整数或浮点数")
    return tuple(x * s for x in t)

print("vec_scalar_mult((1, 2, 3), 2)   =", vec_scalar_mult((1, 2, 3), 2))
print("vec_scalar_mult((1.5, -2), 4)   =", vec_scalar_mult((1.5, -2), 4))
print("vec_scalar_mult((1, 2), 0.5)    =", vec_scalar_mult((1, 2), 0.5))

for bad_args in [([1, 2], 2), ((1, 2), "x"), ((1, "a"), 2)]:
    try:
        vec_scalar_mult(*bad_args)
    except TypeError as exc:
        print(f"vec_scalar_mult({bad_args!r}) → TypeError: {exc}")

vec_scalar_mult((1, 2, 3), 2)   = (2, 4, 6)
vec_scalar_mult((1.5, -2), 4)   = (6.0, -8)
vec_scalar_mult((1, 2), 0.5)    = (0.5, 1.0)
vec_scalar_mult(([1, 2], 2)) → TypeError: 第一个参数必须是元组，得到 list
vec_scalar_mult(((1, 2), 'x')) → TypeError: 标量必须是整数或浮点数，得到 str
vec_scalar_mult(((1, 'a'), 2)) → TypeError: 元组的元素必须都是整数或浮点数


### 4-2 `vec_inner_product(x, y)`

内积 $\sum_{j=1}^{d} x_j y_j$。要求：任一参数不是元组 → `TypeError`；长度不同 → `ValueError`。用 `zip` 成对相乘再求和，返回值显式转成 `float`，保证得到的是浮点数。

In [8]:
def vec_inner_product(x, y):
    if not isinstance(x, tuple) or not isinstance(y, tuple):
        raise TypeError("两个参数都必须是元组")
    if len(x) != len(y):
        raise ValueError(f"两个向量长度不一致：{len(x)} 与 {len(y)}")
    return float(sum(a * b for a, b in zip(x, y)))

print("vec_inner_product((1, 2, 3), (4, 5, 6)) =", vec_inner_product((1, 2, 3), (4, 5, 6)))
print("返回类型:", type(vec_inner_product((1, 2, 3), (4, 5, 6))).__name__)

try:
    vec_inner_product([1, 2], (3, 4))
except TypeError as exc:
    print("传入列表 →", type(exc).__name__, ":", exc)

try:
    vec_inner_product((1, 2), (3, 4, 5))
except ValueError as exc:
    print("长度不同 →", type(exc).__name__, ":", exc)

vec_inner_product((1, 2, 3), (4, 5, 6)) = 32.0
返回类型: float
传入列表 → TypeError : 两个参数都必须是元组
长度不同 → ValueError : 两个向量长度不一致：2 与 3


### 4-3 `check_valid_mx(mx)`

判断矩阵（元组的元组）是否有效，需要同时满足：

1. 外层是元组；
2. 每个元素本身是元组；
3. 所有内层元组长度相同；
4. 每个内层元组的元素都是整数或浮点数。

逐条检查，任一不满足立即返回 `False`。空矩阵 `()` 没有行，三条条件都「无行可违反」，因此视为有效。

In [9]:
def check_valid_mx(mx):
    if not isinstance(mx, tuple):
        return False
    if len(mx) == 0:
        return True
    if not all(isinstance(row, tuple) for row in mx):
        return False
    n = len(mx[0])
    if not all(len(row) == n for row in mx):
        return False
    return all(isinstance(v, (int, float)) and not isinstance(v, bool)
               for row in mx for v in row)

valid_mx = ((1, 2), (3, 4), (5, 6))
print("合法矩阵        :", valid_mx, "→", check_valid_mx(valid_mx))
print("空矩阵 ()       :", check_valid_mx(()))
print("外层是列表      :", check_valid_mx([[1, 2], [3, 4]]))
print("内层是列表      :", check_valid_mx(([1, 2], [3, 4])))
print("行长不一致      :", check_valid_mx(((1, 2), (3, 4, 5))))
print("元素含字符串    :", check_valid_mx(((1, 2), (3, "a"))))
print("单行矩阵        :", check_valid_mx(((1, 2, 3),)))

合法矩阵        : ((1, 2), (3, 4), (5, 6)) → True
空矩阵 ()       : True
外层是列表      : False
内层是列表      : False
行长不一致      : False
元素含字符串    : False
单行矩阵        : True


### 4-4 `mx_vec_mult(mx, v)`

用上一题的 `check_valid_mx` 校验矩阵；向量必须是元组；向量长度必须等于矩阵的列数，否则无法相乘。

结果的第 $i$ 个分量是矩阵第 $i$ 行与向量的内积，返回值是元组。

In [10]:
def mx_vec_mult(mx, v):
    if not check_valid_mx(mx):
        raise TypeError("第一个参数不是合法的矩阵（元组的元组，元素为数值）")
    if not isinstance(v, tuple):
        raise TypeError(f"第二个参数必须是元组，得到 {type(v).__name__}")
    n_cols = len(mx[0]) if mx else 0
    if len(v) != n_cols:
        raise ValueError(f"矩阵列数为 {n_cols}，但向量长度为 {len(v)}")
    return tuple(sum(row[j] * v[j] for j in range(n_cols)) for row in mx)

print("矩阵 :", ((1, 2), (3, 4)))
print("向量 :", (5, 6))
print("结果 :", mx_vec_mult(((1, 2), (3, 4)), (5, 6)))
print("手算 : 第一行 1×5 + 2×6 = 17；第二行 3×5 + 4×6 = 39")

for bad_mx, bad_v in [(((1, 2), (3, 4)), [5, 6]), (((1, 2), (3, 4)), (5, 6, 7)), (([[1, 2]]), (1, 2))]:
    try:
        mx_vec_mult(bad_mx, bad_v)
    except (TypeError, ValueError) as exc:
        print(f"非法输入 → {type(exc).__name__}: {exc}")

矩阵 : ((1, 2), (3, 4))
向量 : (5, 6)
结果 : (17, 39)
手算 : 第一行 1×5 + 2×6 = 17；第二行 3×5 + 4×6 = 39
非法输入 → TypeError: 第二个参数必须是元组，得到 list
非法输入 → ValueError: 矩阵列数为 2，但向量长度为 3
非法输入 → TypeError: 第一个参数不是合法的矩阵（元组的元组，元素为数值）


---

## 5. 更多向量的乐趣

### 5-1 `Vector` 类

初始化参数为「维度」和「元素」。设计要点：

- 维度必须是 `int`（`bool` 虽是 `int` 子类但不合理，一并拒绝）；负数抛 `ValueError`；
- 只给维度、不给元素时，默认构造该维度的**零向量**；
- 元素必须是列表或元组，个数必须等于维度，元素必须是数值；
- 内部用元组保存 `elems`，避免外部通过原列表改动向量内容。

为了后续几道小题能直接测试，这个类把所有要求的方法一次性写全（比较、`dot`、乘法），后面各节分别演示。

In [11]:
class Vector:
    def __init__(self, dim, elems=None):
        if isinstance(dim, bool) or not isinstance(dim, int):
            raise TypeError(f"维度必须是整数，得到 {type(dim).__name__}")
        if dim < 0:
            raise ValueError(f"维度不能为负数：{dim}")

        if elems is None:
            elems = (0,) * dim
        elif not isinstance(elems, (list, tuple)):
            raise TypeError(f"元素必须是列表或元组，得到 {type(elems).__name__}")

        if len(elems) != dim:
            raise ValueError(f"维度为 {dim}，但给出了 {len(elems)} 个元素")
        if not all(isinstance(x, (int, float)) and not isinstance(x, bool) for x in elems):
            raise TypeError("元素必须都是整数或浮点数")

        self.dim = dim
        self.elems = tuple(elems)

    def __repr__(self):
        return f"Vector({self.dim}, {self.elems})"

    def _check_dim(self, other):
        if self.dim != other.dim:
            raise ValueError(f"维度不一致：{self.dim} 与 {other.dim}")

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return self.dim == other.dim and self.elems == other.elems

    def __lt__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        self._check_dim(other)
        return self.elems < other.elems

    def __le__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        self._check_dim(other)
        return self.elems <= other.elems

    def __gt__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        self._check_dim(other)
        return self.elems > other.elems

    def __ge__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        self._check_dim(other)
        return self.elems >= other.elems

    def dot(self, other):
        if not isinstance(other, Vector):
            raise TypeError(f"dot() 的参数必须是 Vector，得到 {type(other).__name__}")
        self._check_dim(other)
        return float(sum(a * b for a, b in zip(self.elems, other.elems)))

    def __mul__(self, other):
        if isinstance(other, Vector):
            self._check_dim(other)
            return Vector(self.dim, tuple(a * b for a, b in zip(self.elems, other.elems)))
        if isinstance(other, (int, float)) and not isinstance(other, bool):
            return Vector(self.dim, tuple(x * other for x in self.elems))
        return NotImplemented

    def __rmul__(self, other):
        return self.__mul__(other)

v = Vector(3, [1, 2, 3])
print("v            :", v)
print("v.dim        :", v.dim)
print("v.elems      :", v.elems)
print("默认零向量    :", Vector(4))

for args in [(2, [1, 2, 3]), (-1, [1]), (2.5, [1, 2]), (2, "ab")]:
    try:
        Vector(*args)
    except (TypeError, ValueError) as exc:
        print(f"Vector{args} → {type(exc).__name__}: {exc}")

v            : Vector(3, (1, 2, 3))
v.dim        : 3
v.elems      : (1, 2, 3)
默认零向量    : Vector(4, (0, 0, 0, 0))
Vector(2, [1, 2, 3]) → ValueError: 维度为 2，但给出了 3 个元素
Vector(-1, [1]) → ValueError: 维度不能为负数：-1
Vector(2.5, [1, 2]) → TypeError: 维度必须是整数，得到 float
Vector(2, 'ab') → TypeError: 元素必须是列表或元组，得到 str


### 5-2 维度与元素应为实例属性还是类属性？

**应为实例属性**（在 `__init__` 中通过 `self.dim`、`self.elems` 赋值），而不是类属性。

理由：

1. **每个向量的数据是独立的**。类属性属于类本身、被所有实例共享。若写 `dim = 0`、`elems = ()` 在类体里，所有实例读到的是同一份数据；只有再通过 `self.dim = ...` 赋值才会在实例上「遮住」类属性，容易造成改一个对象影响全体的错觉。
2. **可变对象共享更危险**。如果类属性写成列表（如 `elems = []`），所有实例会共用同一个列表，`v1.elems.append(...)` 会同时改变 `v2` 的结果。
3. **语义上维度是每个向量的固有属性**，不同维度、不同坐标的向量必须各自保存自己的值。

下面的小实验演示类属性被共享的情况，与 `Vector` 的实例属性形成对照：

In [12]:
class SharedDemo:
    elems = []          # 类属性：所有实例共享同一个列表

    def __init__(self, value):
        self.value = value


a = SharedDemo(1)
b = SharedDemo(2)
a.elems.append("被 a 修改")
print("SharedDemo.elems :", SharedDemo.elems)
print("b.elems          :", b.elems, "← b 没有操作，却跟着变了")

v1, v2 = Vector(2, (1, 2)), Vector(2, (3, 4))
print()
print("Vector 的实例属性互不影响：")
print("v1.elems :", v1.elems)
print("v2.elems :", v2.elems)

SharedDemo.elems : ['被 a 修改']
b.elems          : ['被 a 修改'] ← b 没有操作，却跟着变了

Vector 的实例属性互不影响：
v1.elems : (1, 2)
v2.elems : (3, 4)


### 5-3 比较运算符

- **相等**：坐标完全相同才算相等；维度不同时直接判为 `False`（不是同一个向量），与 `==` 的常规语义一致。
- **大小**：按元组的字典序比较——先比第一个坐标，相同再比第二个，依此类推。直接复用元组自带的比较即可，无需手写循环。
- **维度不同**：题目要求抛出异常，因此 `<`、`<=`、`>`、`>=` 在维度不一致时抛 `ValueError`。

`__eq__` 返回 `NotImplemented` 用于和非 `Vector` 对象比较，让 Python 去尝试对方的实现，而不是强行报错。

In [13]:
a = Vector(2, (2, 4))
b = Vector(2, (2, 5))
c = Vector(2, (2, 4))

print("a =", a, " b =", b, " c =", c)
print()
print("a == c :", a == c)
print("a == b :", a == b)
print("a <  b :", a < b)
print("a <= b :", a <= b)
print("b >  a :", b > a)
print("b >= a :", b >= a)

try:
    Vector(2, (1, 2)) < Vector(3, (1, 2, 3))
except ValueError as exc:
    print()
    print("维度不同比较 →", type(exc).__name__, ":", exc)

a = Vector(2, (2, 4))  b = Vector(2, (2, 5))  c = Vector(2, (2, 4))

a == c : True
a == b : False
a <  b : True
a <= b : True
b >  a : True
b >= a : True

维度不同比较 → ValueError : 维度不一致：2 与 3


### 5-4 `Vector.dot()`

参数必须是 `Vector`（否则 `TypeError`），维度必须一致（否则 `ValueError`），返回浮点数。

In [14]:
u = Vector(3, (1, 2, 3))
w = Vector(3, (4, 5, 6))

print(f"{u}.dot({w}) =", u.dot(w))
print("返回类型:", type(u.dot(w)).__name__)

try:
    u.dot((4, 5, 6))
except TypeError as exc:
    print("传入元组 →", type(exc).__name__, ":", exc)

try:
    u.dot(Vector(2, (1, 2)))
except ValueError as exc:
    print("维度不同 →", type(exc).__name__, ":", exc)

Vector(3, (1, 2, 3)).dot(Vector(3, (4, 5, 6))) = 32.0
返回类型: float
传入元组 → TypeError : dot() 的参数必须是 Vector，得到 tuple
维度不同 → ValueError : 维度不一致：3 与 2


### 5-5 乘法运算符

分三种情况：

- `v * 2`：标量在右，走 `__mul__`，每个分量乘以 2；
- `2 * v`：标量在左，`int.__mul__` 不认识 `Vector`，于是调用 `Vector.__rmul__`，这里直接复用 `__mul__` 的逻辑；
- `v * w`：两个 `Vector` 逐分量相乘（Hadamard 积），结果仍是新的 `Vector`。

两种乘法都返回**新对象**，原向量不变；维度不一致时抛 `ValueError`。

In [15]:
v = Vector(2, (2, 3))
w = Vector(2, (4, 5))

print("v * 2 :", v * 2)
print("2 * v :", 2 * v)
print("v * w :", v * w)
print()
print("原向量保持不变：v =", v, "，w =", w)
print("结果是新对象？", (v * w) is not v and (v * w) is not w)

try:
    v * Vector(3, (1, 2, 3))
except ValueError as exc:
    print()
    print("维度不同相乘 →", type(exc).__name__, ":", exc)

v * 2 : Vector(2, (4, 6))
2 * v : Vector(2, (4, 6))
v * w : Vector(2, (8, 15))

原向量保持不变：v = Vector(2, (2, 3)) ，w = Vector(2, (4, 5))
结果是新对象？ True

维度不同相乘 → ValueError : 维度不一致：2 与 3


---

## 6. 小结

| 部分 | 函数 / 类 | 关键点 |
|---|---|---|
| 字符串 | `is_palindrome` | 过滤非字母数字、统一小写后与逆序比较 |
| 字符串 | `is_abecedarian` | 去空格后检查相邻字符非递减，允许重复 |
| 字符串 | `remove_vowels` | 生成器过滤元音，全元音输入返回空串 |
| 列表 | `list_reverse` | 非列表抛 `TypeError`；切片返回新列表 |
| 列表 | `is_sorted` | 单次扫描 $O(n)$，不复制、不排序 |
| 元组向量 | `vec_scalar_mult` | 类型全检查，返回新元组 |
| 元组向量 | `vec_inner_product` | 非元组 `TypeError`、长度不等 `ValueError` |
| 矩阵 | `check_valid_mx` | 四条件逐条判断，空矩阵视为有效 |
| 矩阵 | `mx_vec_mult` | 复用校验函数，列数与向量长度必须匹配 |
| 类 | `Vector` | 实例属性；字典序比较；`dot` 与标量/逐分量乘法 |